# Cuaderno para evaluar el modelo


## Configuración del cuaderno


In [ ]:
#Instalación de paquetes
!pip install tf_keras tensorflow numpy matplotlib -q
!pip install coral-ordinal

import os

# Forzar uso de Keras 2 para evitar problemas de compatibilidad con STM32Cube.AI
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tf_keras as keras
from tf_keras import layers, Model
import coral_ordinal as coral


# Verificar que estamos en Keras 2
assert int(keras.__version__.split('.')[0]) == 2, "No se está usando la versión de Keras2"
print("Usando Keras2")

from google.colab import drive
drive.mount('/content/drive')

import sys

SRC_PATH = "/content/drive/MyDrive/TFG/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

In [ ]:
# Macros

TEST_PATH = '/content/drive/MyDrive/TFG/dataset/test'
MODEL_PATH = '/content/drive/MyDrive/TFG/modelos'

IMAGE_SIZE = (480, 270)

NUM_BATCHES = 32

LABELS = {'0':'fluido', '1' : 'moderado', '2' : 'denso', '3' : 'saturado'}
NUM_CLASES = len(LABELS)


## Funciones auxiliares

In [ ]:
# Preprocesado de imágenes
from preprocesado import preprocesado, ImageCropY
preprocesado = preprocesado(IMAGE_SIZE)

# Evaluación del modelo
from evaluacion import evaluar_modelo

##Dataset


In [ ]:
# Carga dataset de test
test_ds = keras.utils.image_dataset_from_directory (
    directory = TEST_PATH,
    labels = 'inferred',
    label_mode = 'int',
    batch_size = NUM_BATCHES,
    image_size = IMAGE_SIZE
)

test_ds = test_ds.map(
    lambda x, y: (preprocesado(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

## Evaluación

In [ ]:
# Carga del modelo
CORAL_CUSTOM_OBJECTS = {
    'CoralOrdinal'           : coral.CoralOrdinal,
    'OrdinalCrossEntropy'    : coral.OrdinalCrossEntropy,
    'MeanAbsoluteErrorLabels': coral.MeanAbsoluteErrorLabels,
    'ImageCropY'             : ImageCropY
}

model = keras.models.load_model(
    f'{MODEL_PATH}/modelo_final.keras',
    custom_objects=CORAL_CUSTOM_OBJECTS
)
print("Modelo cargado correctamente")

In [ ]:
evaluar_modelo (model, test_ds, LABELS, MODEL_PATH)